# このNotebookでは、ウェブスクレイピングを使って、大学のイベント情報を集める：
### 各研究科などの情報を集めるのだが、まず医学部のHPから集められる情報を取得する

In [1]:
import requests
from bs4 import BeautifulSoup
import json 
import time
import pandas as pd
import numpy as np
import re 
from urllib.parse import urljoin


In [20]:
url = "https://www.med.tohoku.ac.jp/6122/"
req = requests.get("https://www.med.tohoku.ac.jp/6122/")
req.encoding = req.apparent_encoding 

In [22]:
import urllib.robotparser
# https://qiita.com/Broccolingual/items/aa1f48454b9972b82d63 これを参考にrobots.txtの情報からAgentでクロール可能かどうかを判定する

# robots.txtの読み取り
robots_txt_url = url + 'robot.txt'
rp = urllib.robotparser.RobotFileParser()
rp.set_url(robots_txt_url)
rp.read()

# robots.txtの情報から調査したいURL、User-Agentでクロール可能かを調べる
user_agent = '*'
result = rp.can_fetch(user_agent, url)
print(result)

True


In [100]:
soup = BeautifulSoup(req.content, "html.parser")

In [101]:
# print(soup.prettify())

In [102]:
soup.select("h3")
# soup.select("p")

[<h3>ノーザン・ブリティッシュ・コロンビア大学との学術交流協定調印式を執り行いました</h3>]

In [104]:
soup.select("time")

[<time>2025年12月08日</time>]

In [119]:
tags_ul = soup.find("ul", class_="tags")

if tags_ul:
    # liタグごとのテキストをリスト化して、カンマなどで結合
    # 結果例: "ニュース, イベント"
    tag_text = ", ".join([li.text.strip() for li in tags_ul.find_all('li')])

tag_text

'ニュース'

In [103]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="ja">
 <head>
  <!-- Google tag (gtag.js) -->
  <script async="" src="https://www.googletagmanager.com/gtag/js?id=G-03S2PJQN0Y">
  </script>
  <script>
   window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'G-03S2PJQN0Y');
  gtag('config', 'UA-9434692-1');
  </script>
  <meta charset="utf-8"/>
  <meta content="ie=edge" http-equiv="x-ua-compatible"/>
  <title>
   東北大学大学院医学系研究科・医学部
  </title>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="telephone=no" name="format-detection"/>
  <meta content="東北大学医学系研究科では、広く医学領域の問題解決に挑戦する人材の育成を目指しています。" name="description"/>
  <link href="https://www.med.tohoku.ac.jp/wp-content/themes/medtohoku/common/css/font-awesome.min.css" rel="stylesheet"/>
  <link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@300;500&amp;family=Noto+Sans+JP:wght@400;500;700;900&amp;display=swap" rel="stylesheet"/>
  

In [77]:
article_id = url.split('/')[-2]
article_id

'6122'

In [2]:
base_url = "https://www.med.tohoku.ac.jp/news/?y=2025"
response = requests.get(base_url)
response.encoding = response.apparent_encoding

soup = BeautifulSoup(response.text, 'html.parser')
all_links = soup.find_all('a')

href_list = []

# 3. リンクの抽出とフィルタリング
for link in all_links:
    url = link.get('href')
    
    if url:
        # ★重要: 相対パス（例: /news/detail/）を絶対パス（https://...）に変換
        full_url = urljoin(base_url, url)
        
        if full_url.startswith("https://www.med.tohoku.ac.jp/") and full_url.endswith("/"):
           
            if full_url not in href_list:
                href_list.append(full_url)
print(f"見つかった件数: {len(href_list)}")
for link in href_list:
    print(link)



見つかった件数: 251
https://www.med.tohoku.ac.jp/
https://www.med.tohoku.ac.jp/english/
https://www.med.tohoku.ac.jp/news/
https://www.med.tohoku.ac.jp/category/news/
https://www.med.tohoku.ac.jp/category/event/
https://www.med.tohoku.ac.jp/category/awards/
https://www.med.tohoku.ac.jp/researchlist/
https://www.med.tohoku.ac.jp/category/recruit/
https://www.med.tohoku.ac.jp/6124/
https://www.med.tohoku.ac.jp/6123/
https://www.med.tohoku.ac.jp/6122/
https://www.med.tohoku.ac.jp/6121/
https://www.med.tohoku.ac.jp/6120/
https://www.med.tohoku.ac.jp/6119/
https://www.med.tohoku.ac.jp/6118/
https://www.med.tohoku.ac.jp/6117/
https://www.med.tohoku.ac.jp/6093/
https://www.med.tohoku.ac.jp/6116/
https://www.med.tohoku.ac.jp/6115/
https://www.med.tohoku.ac.jp/6114/
https://www.med.tohoku.ac.jp/6113/
https://www.med.tohoku.ac.jp/6112/
https://www.med.tohoku.ac.jp/6111/
https://www.med.tohoku.ac.jp/6110/
https://www.med.tohoku.ac.jp/6109/
https://www.med.tohoku.ac.jp/6108/
https://www.med.tohoku.ac.jp/

In [3]:
href_list[10]

'https://www.med.tohoku.ac.jp/6122/'

In [4]:
pattern = r'https://www\.med\.tohoku\.ac\.jp/\d+/$'
target_urls = [url for url in href_list if re.match(pattern, url)]
print(f"条件に一致したURL: {len(target_urls)}件")
for url in target_urls:
    print(url)

条件に一致したURL: 243件
https://www.med.tohoku.ac.jp/6124/
https://www.med.tohoku.ac.jp/6123/
https://www.med.tohoku.ac.jp/6122/
https://www.med.tohoku.ac.jp/6121/
https://www.med.tohoku.ac.jp/6120/
https://www.med.tohoku.ac.jp/6119/
https://www.med.tohoku.ac.jp/6118/
https://www.med.tohoku.ac.jp/6117/
https://www.med.tohoku.ac.jp/6093/
https://www.med.tohoku.ac.jp/6116/
https://www.med.tohoku.ac.jp/6115/
https://www.med.tohoku.ac.jp/6114/
https://www.med.tohoku.ac.jp/6113/
https://www.med.tohoku.ac.jp/6112/
https://www.med.tohoku.ac.jp/6111/
https://www.med.tohoku.ac.jp/6110/
https://www.med.tohoku.ac.jp/6109/
https://www.med.tohoku.ac.jp/6108/
https://www.med.tohoku.ac.jp/6107/
https://www.med.tohoku.ac.jp/6104/
https://www.med.tohoku.ac.jp/6106/
https://www.med.tohoku.ac.jp/6105/
https://www.med.tohoku.ac.jp/6103/
https://www.med.tohoku.ac.jp/6102/
https://www.med.tohoku.ac.jp/6101/
https://www.med.tohoku.ac.jp/6100/
https://www.med.tohoku.ac.jp/6084/
https://www.med.tohoku.ac.jp/6099/
htt

In [5]:
targets=target_urls[0:10]

In [11]:
data_list = []
    
for url in targets :
    print(f"スクレイピング中: {url}")
        
    try:
        # サーバー負荷軽減のため1秒待機
        time.sleep(1)

        r = requests.get(url)
        r.encoding = r.apparent_encoding 

        if r.status_code==200:

            soup = BeautifulSoup(r.content, "html.parser")
            article_id = url.split('/')[-2]
            h3_tags = soup.select("h3")
            time_tags = soup.select("time")
            release_date = time_tags[0].text.strip() if time_tags else ""
            

            tags_ul = soup.find("ul", class_="tags")

            if tags_ul:
                # liタグごとのテキストをリスト化して、カンマなどで結合
                # 結果例: "ニュース, イベント"
                tag_text = ", ".join([li.text.strip() for li in tags_ul.find_all('li')])



            if len(h3_tags) >= 2:
                    title = h3_tags[1].text.strip()

            else:
                title = h3_tags[0].text.strip() if h3_tags else "タイトルなし"

        # データをリストに追加
            data_list.append({
                    'Article番号': article_id,
                    'タイトル': title,
                    'リリース日': release_date,
                    'Tag':tag_text,
                    'URL': url
            })
                
        else:
                print(f"記事取得失敗: {res_article.status_code}")


    except Exception as e:
        print(f"エラー発生: {e}")
        continue

if data_list:
    df = pd.DataFrame(data_list)
        
        # 結果を表示
    print("\n--- 取得結果 ---")
    print(df.head())
        
        # CSV保存
    df.to_csv('tohoku_news_articles.csv', index=False, encoding='utf-8-sig')
    print("\n'tohoku_news_articles.csv' に保存しました。")
else:
    print("データが取得できませんでした。")




スクレイピング中: https://www.med.tohoku.ac.jp/6124/
スクレイピング中: https://www.med.tohoku.ac.jp/6123/
スクレイピング中: https://www.med.tohoku.ac.jp/6122/
スクレイピング中: https://www.med.tohoku.ac.jp/6121/
スクレイピング中: https://www.med.tohoku.ac.jp/6120/
スクレイピング中: https://www.med.tohoku.ac.jp/6119/
スクレイピング中: https://www.med.tohoku.ac.jp/6118/
スクレイピング中: https://www.med.tohoku.ac.jp/6117/
スクレイピング中: https://www.med.tohoku.ac.jp/6093/
スクレイピング中: https://www.med.tohoku.ac.jp/6116/

--- 取得結果 ---
  Article番号                                               タイトル        リリース日  \
0      6124         北海道・東北・北信・九州がんプロ4拠点連携シンポジウム開催のお知らせ（1/16開催）  2025年12月10日   
1      6123                                 留学生向け交流イベントを開催しました  2025年12月09日   
2      6122            ノーザン・ブリティッシュ・コロンビア大学との学術交流協定調印式を執り行いました  2025年12月08日   
3      6121                                             タイトルなし                
4      6120  糖尿病代謝・内分泌内科学分野の片桐 秀樹教授、今井 淳太准教授が第62回「ベルツ賞」の1等賞...  2025年12月02日   

     Tag                                 URL  
0   イベント  http

In [12]:
df = pd.read_csv("/Users/yoshizawakazuki/Kazuki_module/Webmining/tohoku_news_articles.csv")

In [13]:
df['リリース日']

0    2025年12月10日
1    2025年12月09日
2    2025年12月08日
3            NaN
4    2025年12月02日
5            NaN
6    2025年12月01日
7    2025年11月27日
8    2025年11月27日
9    2025年11月26日
Name: リリース日, dtype: object

In [15]:
df # タイトルなしはPress Releaseで違うFormatになっているから

,Article番号,タイトル,リリース日,Tag,URL
0,6124,北海道・東北・北信・九州がんプロ4拠点連携シンポジウム開催のお知らせ（1/16開催）,2025年12月10日,イベント,https://www.med.tohoku.ac.jp/6124/
1,6123,留学生向け交流イベントを開催しました,2025年12月09日,ニュース,https://www.med.tohoku.ac.jp/6123/
2,6122,ノーザン・ブリティッシュ・コロンビア大学との学術交流協定調印式を執り行いました,2025年12月08日,ニュース,https://www.med.tohoku.ac.jp/6122/
3,6121,タイトルなし,NaN,ニュース,https://www.med.tohoku.ac.jp/6121/
4,6120,糖尿病代謝・内分泌内科学分野の片桐 秀樹教授、今井 淳太准教授が第62回「ベルツ賞」の1等賞...,2025年12月02日,受賞・表彰,https://www.med.tohoku.ac.jp/6120/
5,6119,タイトルなし,NaN,受賞・表彰,https://www.med.tohoku.ac.jp/6119/
6,6118,東北医学会セミナー開催のお知らせ（1/9開催）,2025年12月01日,イベント,https://www.med.tohoku.ac.jp/6118/
7,6117,【12/2放送予定】NHK総合「未来予測反省会」に医療倫理学分野 浅井 篤教授が出演します,2025年11月27日,ニュース,https://www.med.tohoku.ac.jp/6117/
8,6093,東北大学病院 総合外科 医局秘書の募集,2025年11月27日,採用情報,https://www.med.tohoku.ac.jp/6093/
9,6116,地域がん医療推進センター 事務補佐員の募集,2025年11月26日,採用情報,https://www.med.tohoku.ac.jp/6116/
